### Name: Zehan Li

### Analysis: Doing the Data Cleaning

#### AI uses: didn't using AI for code logic, but using AI for annotation so that teammates could understand better.

In [1]:
import pandas as pd
import os
import re
from pathlib import Path

# Project data root
BASE = "/Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data"
PRE_DIR = Path(BASE) / "pre_AI"
POST_DIR = Path(BASE) / "post_AI"


def infer_event_from_filename(filename: str) -> str:
    """Infer normalized event label from a CSV filename.

    Example:
      guardian_facebook_outage_2021-10.csv -> facebook_2021
      nyt_cloudflare_2019-07-02.csv -> cloudflare_2019
      guardian_google_cloud_2025.csv -> google_cloud_2025
    """
    stem = Path(filename).stem.lower()

    # Remove source prefix
    stem = re.sub(r"^(guardian|nyt)_", "", stem)

    # Remove generic token
    stem = stem.replace("_outage", "")

    # Extract year and base event name
    year_match = re.search(r"(19|20)\d{2}", stem)
    year = year_match.group(0) if year_match else None

    base = re.sub(r"[_-]?(19|20)\d{2}.*$", "", stem).strip("_-")
    if not base:
        base = stem

    return f"{base}_{year}" if year else base


def discover_input_files():
    """Discover all pre/post AI CSV files and attach metadata.

    Excludes post_AI/postmortem_all.csv by design.
    """
    files = []

    # pre_AI files
    for path in sorted(PRE_DIR.glob("*.csv")):
        name = path.name
        source = "guardian" if name.startswith("guardian_") else "nyt" if name.startswith("nyt_") else "unknown"
        files.append({
            "path": str(path),
            "source": source,
            "event": infer_event_from_filename(name),
            "era": "pre_ai",
            "origin_file": name,
        })

    # post_AI files (exclude postmortem_all.csv)
    for path in sorted(POST_DIR.glob("*.csv")):
        name = path.name
        if name == "postmortem_all.csv":
            continue
        source = "guardian" if name.startswith("guardian_") else "nyt" if name.startswith("nyt_") else "unknown"
        files.append({
            "path": str(path),
            "source": source,
            "event": infer_event_from_filename(name),
            "era": "post_ai",
            "origin_file": name,
        })

    return files


FILES = discover_input_files()
print(f"Total discovered files: {len(FILES)}")
print(f"  pre_ai:  {sum(1 for f in FILES if f['era'] == 'pre_ai')}")
print(f"  post_ai: {sum(1 for f in FILES if f['era'] == 'post_ai')}")


Total discovered files: 20
  pre_ai:  10
  post_ai: 10


In [2]:
def load_guardian(path):
    """Load Guardian CSV and standardize columns."""
    df = pd.read_csv(path)

    if df.empty:
        return pd.DataFrame(columns=["pub_date", "title", "section", "web_url", "trail_text", "body_text"])

    # Flexible mapping for possible column variants
    col_title = "title" if "title" in df.columns else "webTitle" if "webTitle" in df.columns else None
    col_trail = "trailText" if "trailText" in df.columns else "trail_text" if "trail_text" in df.columns else None
    col_body = "bodyText" if "bodyText" in df.columns else "body_text" if "body_text" in df.columns else None

    out = pd.DataFrame({
        "pub_date": df["pub_date"] if "pub_date" in df.columns else pd.NaT,
        "title": df[col_title] if col_title else "",
        "section": df["section"] if "section" in df.columns else "",
        "web_url": df["web_url"] if "web_url" in df.columns else "",
        "trail_text": df[col_trail] if col_trail else "",
        "body_text": df[col_body] if col_body else "",
    })
    return out


def load_nyt(path):
    """Load NYT CSV and standardize columns."""
    df = pd.read_csv(path)

    if df.empty:
        return pd.DataFrame(columns=["pub_date", "title", "section", "web_url", "trail_text", "body_text"])

    # Handle different NYT column names
    date_col = "pub_datetime_utc" if "pub_datetime_utc" in df.columns else "pub_date"
    title_col = "headline" if "headline" in df.columns else "title"
    snippet_col = next(
        (c for c in ["snippet_or_abstract", "snippet", "abstract"] if c in df.columns),
        None
    )
    lead_col = "lead_paragraph" if "lead_paragraph" in df.columns else None

    # Combine snippet + lead_paragraph as body_text (best we can get from NYT)
    snippet = df[snippet_col].fillna("") if snippet_col else pd.Series([""] * len(df))
    lead = df[lead_col].fillna("") if lead_col else pd.Series([""] * len(df))
    body = (snippet.astype(str) + " " + lead.astype(str)).str.strip()

    out = pd.DataFrame({
        "pub_date": df[date_col] if date_col in df.columns else pd.NaT,
        "title": df[title_col] if title_col in df.columns else "",
        "section": df["section"] if "section" in df.columns else "",
        "web_url": df["web_url"] if "web_url" in df.columns else "",
        "trail_text": snippet,  # use snippet as trail_text equivalent
        "body_text": body,
    })
    return out


dfs = []
for info in FILES:
    path = info["path"]

    if not os.path.exists(path):
        print(f"  MISSING: {path}")
        continue

    if os.path.getsize(path) == 0:
        print(f"  SKIP empty file: {path}")
        continue

    try:
        if info["source"] == "guardian":
            df = load_guardian(path)
        elif info["source"] == "nyt":
            df = load_nyt(path)
        else:
            print(f"  SKIP unknown source: {path}")
            continue

        # Add metadata
        df["source"] = info["source"]
        df["event"] = info["event"]
        df["era"] = info["era"]
        df["origin_file"] = info["origin_file"]

        dfs.append(df)
        print(f"  ✓ {info['origin_file']}: {len(df)} rows")

    except Exception as e:
        print(f"  ERROR {info['origin_file']}: {e}")

if not dfs:
    raise RuntimeError("No input files were loaded. Check pre_AI/post_AI directories and CSV formats.")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal rows before cleaning: {len(df_all)}")


  ✓ guardian_aws_2021.csv: 2 rows
  ✓ guardian_facebook_outage_2021-10.csv: 139 rows
  ✓ guardian_fastly_2021.csv: 19 rows
  ✓ guardian_google_2020.csv: 5 rows
  ✓ nyt_aws_2021.csv: 1 rows
  ✓ nyt_cloudflare_2019-07-02.csv: 1 rows
  ✓ nyt_facebook_outage_2021-10-04.csv: 15 rows
  ✓ nyt_facebook_outage_2021-10.csv: 85 rows
  ERROR nyt_fastly_2021.csv: No columns to parse from file
  ✓ nyt_google_2020.csv: 6 rows
  ✓ guardian_aws_2025.csv: 35 rows
  ✓ guardian_azure_2025.csv: 8 rows
  ✓ guardian_cloudflare_outage_2025-11-17_2025-1-20.csv: 2 rows
  ✓ guardian_crowdstrike_2024.csv: 64 rows
  ✓ guardian_google_cloud_2025.csv: 67 rows
  ✓ nyt_aws_2025.csv: 16 rows
  ERROR nyt_azure_2025.csv: No columns to parse from file
  ✓ nyt_cloudflare_2025-11-18.csv: 1 rows
  ✓ nyt_crowdstrike_2024.csv: 40 rows
  ERROR nyt_google_cloud_2025.csv: No columns to parse from file

Total rows before cleaning: 506


In [3]:
print("=== Cleaning ===")

# 1. Standardize dates
df_all["pub_date"] = pd.to_datetime(df_all["pub_date"], utc=True, errors="coerce")

# 2. Drop duplicates by URL (same article from different queries)
before = len(df_all)
df_all = df_all.drop_duplicates(subset="web_url", keep="first")
print(f"Dedup by URL: {before} -> {len(df_all)} (removed {before - len(df_all)})")

# 3. Drop rows with no title
before = len(df_all)
df_all = df_all.dropna(subset=["title"])
df_all = df_all[df_all["title"].str.strip() != ""]
print(f"Drop empty titles: {before} -> {len(df_all)}")

# 4. Create combined text field for analysis
#    Guardian: use body_text (full text)
#    NYT: use body_text (snippet + lead_paragraph)
df_all["text_for_analysis"] = df_all["body_text"].fillna("").astype(str)
df_all["text_length"] = df_all["text_for_analysis"].str.len()
df_all["has_full_text"] = df_all["text_length"] > 200  # rough threshold

# 5. Flag likely irrelevant articles
#    (articles that don't mention outage-related terms at all)
outage_keywords = [
    "outage", "down", "disruption", "failure", "crash", "error",
    "offline", "interrupt", "glitch", "broke", "unavailable",
    "blue screen", "grounded", "cancelled",
]

def is_relevant(row):
    """Check if article is likely about an outage."""
    text = f"{row['title']} {row['text_for_analysis']}".lower()
    return any(kw in text for kw in outage_keywords)

df_all["likely_relevant"] = df_all.apply(is_relevant, axis=1)
irrelevant = (~df_all["likely_relevant"]).sum()
print(f"Flagged as possibly irrelevant: {irrelevant}")
print("  (NOT auto-removed — review these manually)")

# Show irrelevant articles for manual review
if irrelevant > 0:
    print("\n  Sample irrelevant titles:")
    sample = df_all[~df_all["likely_relevant"]]["title"].head(10)
    for t in sample:
        print(f"    - {t[:80]}")



=== Cleaning ===
Dedup by URL: 506 -> 484 (removed 22)
Drop empty titles: 484 -> 484
Flagged as possibly irrelevant: 162
  (NOT auto-removed — review these manually)

  Sample irrelevant titles:
    - ‘Optimism is the only way forward’: the exhibition that imagines our future
    - Met officers investigated over Couzens WhatsApp group are still on duty
    - Evidence of ‘vulgar and sexist’ WhatsApp texts ignored, says ex-Met detective
    - How we met: ‘I sent him a Facebook message by accident’
    - How losing a friend to misinformation drove Facebook whistleblower 
    - Facebook putting profit before public good, says whistleblower Frances Haugen
    - ‘Congress will be taking action’: key takeaways from the Facebook whistleblower 
    - First Thing: Facebook ‘harms children, stokes division, weakens democracy’
    - ‘I might delete it’: Facebook’s problem with younger users
    - Facebook whistleblower testimony should prompt new oversight – Schiff


In [4]:
print("=" * 60)
print("MERGED DATASET SUMMARY")
print("=" * 60)

print(f"\nTotal articles: {len(df_all)}")
print(f"Likely relevant: {df_all['likely_relevant'].sum()}")
print(f"Has full text (>200 chars): {df_all['has_full_text'].sum()}")

print("\n--- By Era ---")
print(df_all.groupby("era").agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
    likely_relevant=("likely_relevant", "sum"),
))

print("\n--- By Event ---")
print(df_all.groupby(["era", "event"]).agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
    likely_relevant=("likely_relevant", "sum"),
))

print("\n--- By Source ---")
print(df_all.groupby(["source"]).agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
    avg_text_len=("text_length", "mean"),
))

print("\n--- By Era x Source ---")
print(df_all.groupby(["era", "source"]).agg(
    articles=("title", "count"),
    has_full_text=("has_full_text", "sum"),
))

MERGED DATASET SUMMARY

Total articles: 484
Likely relevant: 322
Has full text (>200 chars): 405

--- By Era ---
         articles  has_full_text  likely_relevant
era                                              
post_ai       226            174              161
pre_ai        258            231              161

--- By Event ---
                           articles  has_full_text  likely_relevant
era     event                                                      
post_ai aws_2025                 51             35               34
        azure_2025                4              4                2
        crowdstrike_2024        104             68               78
        google_cloud_2025        67             67               47
pre_ai  aws_2021                  3              2                1
        cloudflare_2019           1              1                1
        facebook_2021           224            204              135
        fastly_2021              19             19       

In [5]:
# Full merged dataset
df_all.to_csv(f"{BASE}/merged_all_articles.csv", index=False)
print(f"Saved: {BASE}/merged_all_articles.csv ({len(df_all)} rows)")

# Only relevant articles
df_relevant = df_all[df_all["likely_relevant"]].copy()
df_relevant.to_csv(f"{BASE}/merged_relevant_articles.csv", index=False)
print(f"Saved: {BASE}/merged_relevant_articles.csv ({len(df_relevant)} rows)")

# Guardian-only (full text, for primary analysis)
df_guardian = df_relevant[df_relevant["source"] == "guardian"].copy()
df_guardian.to_csv(f"{BASE}/merged_guardian_fulltext.csv", index=False)
print(f"Saved: {BASE}/merged_guardian_fulltext.csv ({len(df_guardian)} rows)")

# Separate postmortem (don't mix with news)
print(f"\nNote: postmortem_all.csv kept separate — use as technical ground truth")

Saved: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/merged_all_articles.csv (484 rows)
Saved: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/merged_relevant_articles.csv (322 rows)
Saved: /Users/zehanli/Course_Uchicago/Winter_2025/MACS30122/final-project-error-502/data/merged_guardian_fulltext.csv (269 rows)

Note: postmortem_all.csv kept separate — use as technical ground truth


In [6]:
print("=== Sample Articles Per Event ===\n")
for event in df_all["event"].unique():
    subset = df_all[df_all["event"] == event]
    print(f"\n--- {event} ({len(subset)} articles) ---")
    sample = subset.head(3)
    for _, row in sample.iterrows():
        print(f"  [{row['source']}] {row['title'][:70]}")
        print(f"    text_length={row['text_length']}, relevant={row['likely_relevant']}")

=== Sample Articles Per Event ===


--- aws_2021 (3 articles) ---
  [guardian] Amazon Web Services outage hits sites and apps such as IMDb and Tinder
    text_length=2666, relevant=True
  [guardian] ‘Optimism is the only way forward’: the exhibition that imagines our f
    text_length=8222, relevant=False
  [nyt] Crypto Moguls Prepare for Capitol Hill
    text_length=87, relevant=False

--- facebook_2021 (224 articles) ---
  [guardian] Jessica Rowe podcast turns sour over Pauline Hanson interview on ‘why 
    text_length=6981, relevant=True
  [guardian] Don’t say ‘privilege’: can the left find better words for talking with
    text_length=14860, relevant=True
  [guardian] Boris Johnson doesn’t fear Labour. His biggest problem will be his own
    text_length=5486, relevant=True

--- fastly_2021 (19 articles) ---
  [guardian] Rural house prices in England and Wales rise twice as fast as in citie
    text_length=4208, relevant=True
  [guardian] Vroom or bust: is Fast & Furious the ultimat